# Computação Gráfica – Projeto 2
Construção de um cenário 3D com ambiente interno e externo utilizando modelos `.obj`, texturas e câmera móvel.

Feito por:

* Nome completo --- NUSP
* Nome completo --- NUSP

Lista dos controles:

> * **[W], [A], [S], [D]:** movimentação da câmera.
> * **Mouse:** direção da câmera.
> * **Scroll:** controle do campo de visão.
> * **[P]:** alternar visualização da malha poligonal.
> * **[ESC]:** fechar a janela.
> * **[teclas a definir]:** transformações independentes de escala, rotação e translação.


---
## Instalação e inicialização

### Primeiro, vamos importar as bibliotecas necessárias.

In [951]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [952]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


---
## Shaders

### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [953]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

---
## Carregamento de modelos e texturas

### Carregando Modelos (vértices e texturas) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file

In [954]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

### Vamos carregar cada modelo e definir funções para desenhá-los

#### Modelos dos objetos
- x
- y
- ...

In [955]:
# 1) Ambiente interno ---------------------------------------------------------------------------------------

verticeInicial_gato, quantosVertices_gato = load_obj_and_texture(
    'objetos/interno/gato/gato.obj',
    ['objetos/interno/gato/gato.jpg'] 
)

def desenha_gato(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_gato, quantosVertices_gato)

# 2) Ambiente externo ---------------------------------------------------------------------------------------



Processando modelo objetos/interno/gato/gato.obj. Vertice inicial: 0
Processando modelo objetos/interno/gato/gato.obj. Vertice final: 317592
0


#### Modelos do ambiente
- Skybox
- Chão externo
- Chão interno

In [956]:
# Skybox ---------------------------------------------------------------------------------------

# Os shaders específicos da skybox servem para evitar as "costuras" nas divisões das faces do cubo.
skyboxVertexShaderSource = """
#version 330 core

attribute vec3 position;
varying vec3 out_texture;

uniform mat4 model;
uniform mat4 view;
uniform mat4 projection;

void main(){
    out_texture = position;
    vec4 pos = projection * view * model * vec4(position, 1.0);
    gl_Position = pos.xyww;
}
"""

skyboxFragmentShaderSource = """
#version 330 core

varying vec3 out_texture;
uniform samplerCube skybox;

void main(){
    gl_FragColor = texture(skybox, out_texture);
}
"""

skyboxVertexShader = glCreateShader(GL_VERTEX_SHADER)
glShaderSource(skyboxVertexShader, skyboxVertexShaderSource)
glCompileShader(skyboxVertexShader)

skyboxFragmentShader = glCreateShader(GL_FRAGMENT_SHADER)
glShaderSource(skyboxFragmentShader, skyboxFragmentShaderSource)
glCompileShader(skyboxFragmentShader)

skyboxProgram = glCreateProgram()
glAttachShader(skyboxProgram, skyboxVertexShader)
glAttachShader(skyboxProgram, skyboxFragmentShader)
glLinkProgram(skyboxProgram)

glDeleteShader(skyboxVertexShader)
glDeleteShader(skyboxFragmentShader)

glUseProgram(skyboxProgram)
loc_skybox = glGetUniformLocation(skyboxProgram, "skybox")
glUniform1i(loc_skybox, 0)
ourShader.use()

verticeInicial_skybox = len(vertices_list)

skybox_vertices = [
    (-1,  1, -1), (-1, -1, -1), (1, -1, -1),
    (1, -1, -1), (1,  1, -1), (-1,  1, -1),

    (-1, -1,  1), (-1, -1, -1), (-1,  1, -1),
    (-1,  1, -1), (-1,  1,  1), (-1, -1,  1),

    (1, -1, -1), (1, -1,  1), (1,  1,  1),
    (1,  1,  1), (1,  1, -1), (1, -1, -1),

    (-1, -1,  1), (-1,  1,  1), (1,  1,  1),
    (1,  1,  1), (1, -1,  1), (-1, -1,  1),

    (-1,  1, -1), (1,  1, -1), (1,  1,  1),
    (1,  1,  1), (-1,  1,  1), (-1,  1, -1),

    (-1, -1, -1), (-1, -1,  1), (1, -1, -1),
    (1, -1, -1), (-1, -1,  1), (1, -1,  1)
]

for vertex in skybox_vertices:
    vertices_list.append(vertex)
    textures_coord_list.append((0, 0))

quantosVertices_skybox = len(vertices_list) - verticeInicial_skybox

skybox_faces = [
    'objetos/ambiente/Skybox/posx.jpg',
    'objetos/ambiente/Skybox/negx.jpg',
    'objetos/ambiente/Skybox/posy.jpg',
    'objetos/ambiente/Skybox/negy.jpg',
    'objetos/ambiente/Skybox/posz.jpg',
    'objetos/ambiente/Skybox/negz.jpg'
]

textureId_skybox = glGenTextures(1)
glBindTexture(GL_TEXTURE_CUBE_MAP, textureId_skybox)

for i in range(len(skybox_faces)):
    img = Image.open(skybox_faces[i])
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, 1)
    glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

def desenha_skybox(t_x, t_y, t_z, textureId):

    glDepthFunc(GL_LEQUAL)
    glUseProgram(skyboxProgram)

    mat_model = model(0, 0, 0, 0, t_x, t_y, t_z, 50, 50, 50)
    loc_model = glGetUniformLocation(skyboxProgram, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    mat_view = view()
    loc_view = glGetUniformLocation(skyboxProgram, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(skyboxProgram, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices_skybox = glGetAttribLocation(skyboxProgram, "position")
    glEnableVertexAttribArray(loc_vertices_skybox)
    glVertexAttribPointer(loc_vertices_skybox, 3, GL_FLOAT, False, stride, offset)

    glBindTexture(GL_TEXTURE_CUBE_MAP, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_skybox, quantosVertices_skybox)

    glDepthFunc(GL_LESS)
    ourShader.use()


# Chao externo ---------------------------------------------------------------------------------------

verticeInicial_chao_ext = len(vertices_list)
textureId_chao_ext = glGenTextures(1)

vertices_chao_ext = [
    (-300, -0.5, -300), (300, -0.5, -300), (300, -0.5, 300),
    (-300, -0.5, -300), (300, -0.5, 300), (-300, -0.5, 300)
]

textures_chao_ext = [
    (0, 0), (60, 0), (60, 60),
    (0, 0), (60, 60), (0, 60)
]

for vertex in vertices_chao_ext:
    vertices_list.append(vertex)

for texture_coord in textures_chao_ext:
    textures_coord_list.append(texture_coord)

load_texture_from_file(textureId_chao_ext, 'objetos/ambiente/Chao_ext/chao_neve.jpg')

quantosVertices_chao_ext = len(vertices_list) - verticeInicial_chao_ext

def desenha_chao_ext(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_chao_ext, quantosVertices_chao_ext)


# Cabana ---------------------------------------------------------------------------------------

verticeInicial_cabana = len(vertices_list)
textureId_cabana = glGenTextures(1)

modelo_cabana = load_model_from_file('objetos/ambiente/Cabana/Snow covered CottageOBJ.obj')

for face in modelo_cabana['faces']:
    for i in range(1, len(face[0]) - 1):
        for vertice_id in [face[0][0], face[0][i], face[0][i + 1]]:
            vertices_list.append(modelo_cabana['vertices'][vertice_id - 1])
        for texture_id in [face[1][0], face[1][i], face[1][i + 1]]:
            textures_coord_list.append(modelo_cabana['texture'][texture_id - 1])

load_texture_from_file(textureId_cabana, 'objetos/ambiente/Cabana/Cottage Texture.jpg')

quantosVertices_cabana = len(vertices_list) - verticeInicial_cabana

def desenha_cabana(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_cabana, quantosVertices_cabana)


# Chao interno ---------------------------------------------------------------------------------------

verticeInicial_chao_int = len(vertices_list)
textureId_chao_int = glGenTextures(1)

vertices_chao_int = [
    (-28, -2.35, -58), (28, -2.35, -58), (28, -2.35, 38),
    (-28, -2.35, -58), (28, -2.35, 38), (-28, -2.35, 38)
]

textures_chao_int = [
    (0, 0), (6, 0), (6, 10),
    (0, 0), (6, 10), (0, 10)
]

for vertex in vertices_chao_int:
    vertices_list.append(vertex)

for texture_coord in textures_chao_int:
    textures_coord_list.append(texture_coord)

load_texture_from_file(textureId_chao_int, 'objetos/ambiente/Chao_int/chao_madeira.jpg')

quantosVertices_chao_int = len(vertices_list) - verticeInicial_chao_int

def desenha_chao_int(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textureId)

    glDrawArrays(GL_TRIANGLES, verticeInicial_chao_int, quantosVertices_chao_int)



2
3
4


---
## Envio ao buffer e alocação na GPU

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar dois slots (buffers): um para os vértices e outro para as texturas.

In [957]:
buffer_VBO = glGenBuffers(2)

### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [958]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [959]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


---
## Eventos de teclado e câmera

### Eventos para modificar a posição da câmera.

* Teclas A, S, D e W para movimentação no espaço tridimensional
* Posição do mouse para "direcionar" a câmera

In [960]:
'''
Variáveis globais das transformações por eventos de teclado
'''

#cameraPos   = glm.vec3(0.0,  0.0,  1.0);
#cameraFront = glm.vec3(0.0,  0.0, -1.0);
#cameraUp    = glm.vec3(0.0,  1.0,  0.0);

# camera
cameraPos   = glm.vec3(0.0, 0.0, 3.0)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = -90.0	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  0.0
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

# limites invisiveis da cena
limite_camera_x_min = -45.0
limite_camera_x_max = 45.0
limite_camera_y_min = 0.0
limite_camera_y_max = 18.0
limite_camera_z_min = -45.0
limite_camera_z_max = 45.0



In [961]:
'''
Definição dos eventos de teclado
'''

def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp, polygonal_mode

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    
    cameraSpeed = 50 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    cameraPos.x = max(limite_camera_x_min, min(cameraPos.x, limite_camera_x_max))
    cameraPos.y = max(limite_camera_y_min, min(cameraPos.y, limite_camera_y_max))
    cameraPos.z = max(limite_camera_z_min, min(cameraPos.z, limite_camera_z_max))

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    # ========================================================
    # PROJETO 2 — ADICIONE AQUI OS EVENTOS DAS TRANSFORMAÇÕES
    # ========================================================
    # O enunciado pede escala, rotação e translação em modelos diferentes,
    # acionadas por teclado de forma independente.
    #
    # Sugestão: defina variáveis globais na célula "Variáveis das transformações"
    # e altere essas variáveis aqui. Depois use os valores no loop principal.
    #
    # Exemplo de organização, mantendo a lógica do professor:
    #
    # global angulo_modelo_rotacao, escala_modelo_escala, deslocamento_modelo_translacao
    #
    # if key == glfw.KEY_R and (action == glfw.PRESS or action == glfw.REPEAT):
    #     angulo_modelo_rotacao += 5.0
    #
    # if key == glfw.KEY_T and (action == glfw.PRESS or action == glfw.REPEAT):
    #     deslocamento_modelo_translacao += 0.1
    #
    # if key == glfw.KEY_Y and (action == glfw.PRESS or action == glfw.REPEAT):
    #     escala_modelo_escala += 0.05
    #
    # Também é aqui que você pode restringir cameraPos para não sair
    # dos limites do terreno/céu/skybox.
        

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    

    

glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

---
## Matrizes de transformação

### Matrizes Model, View e Projection

In [962]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 500.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

---
## Janela final

### Nesse momento, nós exibimos a janela!


In [963]:
glfw.show_window(window)

### Loop principal da janela.

In [964]:
glEnable(GL_DEPTH_TEST) ### importante para 3D
polygonal_mode = False 

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK,GL_FILL)

    
    desenha_skybox(cameraPos.x, cameraPos.y, cameraPos.z, textureId_skybox)

    '''
    Guia das funções desenha_objeto:
        1. ângulo de rotação (em graus)
        2. eixos x, y, z ('1' aplica o ângulo de rotação integralmente, '0' não aplica rotação)
        3. deslocamento em x, y, z 
        4. escala em x, y, z
        5. id da textura a ser aplicada
    '''
    desenha_chao_ext(0, 0, 0, 0, 0, 0, 0, 1, 1, 1, textureId_chao_ext)
    desenha_cabana(225, 0, 1, 0, -7, -0.5, -12, 0.08, 0.08, 0.08, textureId_cabana)
    desenha_chao_int(225, 0, 1, 0, -7.5, 0.0001, -12.5, 0.079, 0.08, 0.075, textureId_chao_int)
    desenha_gato(-90, 1, 0, 0, -7.5, -0.2, -12, 0.015, 0.015, 0.015, 0)
    
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)

glfw.terminate()